<a href="https://colab.research.google.com/github/niclohan/CalculoNumerico/blob/main/Atividade_19/10_Gauss_Seidel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cálculo Numérico — Lista de Exercícios: Gauss–Seidel**
**Nome:** Nichollas Lohan — **RA:** 164928

Sistema:
$$\begin{cases} 10x - y + 2z = 6 \\ -x + 11y - z = 22 \\ 2x - y + 10z = -10 \end{cases}$$


## Bibliotecas
Só `numpy` (para guardar a matriz e conferir a resposta) e `matplotlib` (para os gráficos).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Questão 5 — Método de Gauss–Seidel

**Ideia do método.** Isolamos uma incógnita em cada equação:

$$x^{(k+1)} = \frac{6 + y^{(k)} - 2z^{(k)}}{10}, \quad
y^{(k+1)} = \frac{22 + x^{(k+1)} + z^{(k)}}{11}, \quad
z^{(k+1)} = \frac{-10 - 2x^{(k+1)} + y^{(k+1)}}{10}$$

A diferença para o método de Jacobi: assim que calculamos um valor novo, **já o usamos** nas contas seguintes
da mesma iteração (por exemplo, o $y^{(k+1)}$ usa o $x^{(k+1)}$ recém-calculado).

**Critério de parada:** $\|x^{(k)} - x^{(k-1)}\|_\infty = \max_i |x_i^{(k)} - x_i^{(k-1)}| < 10^{-5}$.

**Como o código faz isso, de forma geral:** para cada linha $i$, o novo $x_i$ é

$$x_i = \frac{b_i - \sum_{j \neq i} a_{ij} x_j}{a_{ii}}$$

e o vetor `x` é atualizado na hora, então os $x_j$ já atualizados entram nas contas seguintes automaticamente.

In [ ]:
def gauss_seidel(A, b, x0, tol=1e-5, max_iter=100):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    x = np.array(x0, dtype=float)
    n = len(b)

    historico = [x.copy()]   # guarda x^(0), x^(1), x^(2), ...
    erros = [None]           # erro de cada iteração (não existe para k = 0)

    for k in range(1, max_iter + 1):
        x_antigo = x.copy()

        for i in range(n):
            # soma de a_ij * x_j para todo j diferente de i
            # (os x_j com j < i já foram atualizados nesta iteração)
            soma = A[i] @ x - A[i, i] * x[i]
            x[i] = (b[i] - soma) / A[i, i]

        erro = np.max(np.abs(x - x_antigo))   # norma do máximo
        historico.append(x.copy())
        erros.append(erro)

        if erro < tol:
            break

    return x, historico, erros

**Aplicando ao nosso sistema**, com $x^{(0)} = (0, 0, 0)$:

In [ ]:
A = [[10, -1, 2],
     [-1, 11, -1],
     [2, -1, 10]]
b = [6, 22, -10]

x, historico, erros = gauss_seidel(A, b, x0=[0, 0, 0], tol=1e-5)

print("Número de iterações:", len(historico) - 1)
print("Solução aproximada: x = %.6f, y = %.6f, z = %.6f" % tuple(x))

**Conferência** com `numpy.linalg.solve` (solução exata, a menos de arredondamento):

In [ ]:
x_exato = np.linalg.solve(A, b)
print("numpy.linalg.solve:", x_exato)
print("Diferença máxima:  ", np.max(np.abs(x - x_exato)))

## Questão 6 — Tabela das iterações

Cada linha mostra a aproximação $(x^{(k)}, y^{(k)}, z^{(k)})$ e o erro entre iterações consecutivas.

In [ ]:
print(f"{'k':>2} | {'x(k)':>10} | {'y(k)':>10} | {'z(k)':>10} | {'||x(k)-x(k-1)||':>16}")
print("-" * 60)

for k, (xk, erro) in enumerate(zip(historico, erros)):
    erro_txt = "-" if erro is None else f"{erro:.3e}"
    print(f"{k:>2} | {xk[0]:>10.6f} | {xk[1]:>10.6f} | {xk[2]:>10.6f} | {erro_txt:>16}")

## Questão 7 — Gráficos

**a)** Evolução de $x^{(k)}$, $y^{(k)}$ e $z^{(k)}$: as três curvas se aproximam de valores constantes, ou seja,
o método está **convergindo**.

**b)** Erro entre iterações em escala logarítmica. Uma reta descendo indica convergência com
**taxa aproximadamente constante** (cada iteração reduz o erro por um fator parecido).

In [ ]:
ks = list(range(len(historico)))
xs = [h[0] for h in historico]
ys = [h[1] for h in historico]
zs = [h[2] for h in historico]

# a) aproximações em função de k
plt.figure(figsize=(6, 4))
plt.plot(ks, xs, "o-", label="x(k)")
plt.plot(ks, ys, "s-", label="y(k)")
plt.plot(ks, zs, "^-", label="z(k)")
plt.xlabel("iteração k")
plt.ylabel("valor da aproximação")
plt.title("Evolução das aproximações")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# b) erro entre iterações (escala log). O erro só existe a partir de k = 1.
plt.figure(figsize=(6, 4))
plt.semilogy(ks[1:], erros[1:], "o-", color="crimson")
plt.axhline(1e-5, color="gray", linestyle="--", label="tolerância 1e-5")
plt.xlabel("iteração k")
plt.ylabel("||x(k) - x(k-1)||∞")
plt.title("Erro entre iterações (escala logarítmica)")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.show()

## Conclusão

O método de Gauss–Seidel convergiu em poucas iterações e a solução obtida coincide com a de
`numpy.linalg.solve`. Isso era esperado: a matriz $A$ é **diagonalmente dominante por linhas**
($|10| > |-1| + |2|$, $|11| > |-1| + |-1|$, $|10| > |2| + |-1|$), o que garante a convergência do método
para qualquer aproximação inicial.